In [1]:
import pandas as pd
import keras
from keras.layers import Input, Flatten, Dense, Lambda, Reshape, Activation, ReLU
from keras.models import Model
from keras import backend as K
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, confusion_matrix
import tensorflow as tf
import keras
import keras_tuner
from keras_tuner import HyperModel
from keras_tuner import HyperParameters
from keras_tuner.tuners import BayesianOptimization
from keras_tuner.engine.hyperparameters import HyperParameters

In [5]:
# Cargar el dataset
# Ficheros V1
#file_train = r"C:\Users\Usuario\Desktop\LSTM2\datos\OLD\datos_entrenamiento_serie_sat3.xlsx"
#file_test = r"C:\Users\Usuario\Desktop\LSTM2\datos\OLD\datos_validacion_serie_sat3.xlsx"

# Ficheros V2
file_train = r"C:\Users\Usuario\Desktop\LSTM2\datos\N\N_datos_entrenamiento_serie_sat3.xlsx"
file_test = r"C:\Users\Usuario\Desktop\LSTM2\datos\N\N_datos_validacion_serie_sat3.xlsx"

X_train = pd.read_excel(file_train, sheet_name = "Sheet1", index_col = 0)
X_test = pd.read_excel(file_test, sheet_name = "Sheet1", index_col = 0)

In [6]:
# Preparar los datos de entrenamiento
X_train = X_train[X_train.anomaly == 0]  ## Usaremos únicamente la clase 0 (transacciones normales)
Y_train = X_train['anomaly'] #identificación de si los datos de train son anómalos o no
X_train = X_train.drop(columns = ["anomaly"], axis=1)
X_train_columns = X_train.columns
X_train = X_train.values

print(X_train.shape, X_test.shape, Y_train.shape)
indices = np.arange(len(X_train))
print(indices.shape)

(2312, 35) (1156, 36) (2312,)
(2312,)


In [7]:
# Dividimos los datos: 60% train, 40% validación
train_size1 = int(len(X_train) * 0.5)

# Split into training and test sets
if len(X_train)%2 == 0:
    X_val = X_train[train_size1:]
    Y_val = Y_train[train_size1:]
else:
    X_val = X_train[train_size1+1:]
    Y_val = Y_train[train_size1+1:]
    
X_train = X_train[:train_size1]
Y_train = Y_train[:train_size1]

In [13]:
X_train = X_train.reshape(1, X_train.shape[0], X_train.shape[1])
X_val = X_val.reshape(1, X_val.shape[0], X_val.shape[1])

ValueError: cannot reshape array of size 40460 into shape (1,1,1156)

In [9]:
#definimos el modelo con los hiperparámetros a optimizar (en este caso, learning rate, batch size, funciones de activación intermedias y función de coste)
def build_model(hp):
    hp_learning_rate = hp.Float('lr', min_value = 0.00001, max_value = 0.001, step = 10, sampling='log')
    hp_batch_size = hp.Int('batch_size', min_value=32, max_value=128, step=2, sampling='log')
    hp_loss_funct = hp.Choice('loss_funct', ['mse', 'mae', 'msle'])
    
    hp_act_funct_out = hp.Choice('act_out', ['linear', 'sigmoid'])
    
    hp_units = hp.Int('units',min_value=32, max_value=128, step=2, sampling='log')
    # hp_dropout = hp.Float('dropout', min_value=0, max_value=0.5, step=0.1)

    model = tf.keras.models.Sequential([
      tf.keras.layers.LSTM(hp_units, return_sequences=True),
      tf.keras.layers.LSTM(hp_units, return_sequences=False),
      tf.keras.layers.RepeatVector(X_train.shape[1]),
      tf.keras.layers.LSTM(hp_units, return_sequences=True),
      tf.keras.layers.LSTM(hp_units, return_sequences=True),
      tf.keras.layers.TimeDistributed(Dense(X_train.shape[2], activation=hp_act_funct_out)),
    ])

    model.compile(keras.optimizers.Adam(learning_rate=hp_learning_rate), loss=hp_loss_funct)

    return model

In [10]:
build_model(keras_tuner.HyperParameters())

<Sequential name=sequential, built=False>

In [11]:
#iniciamos la optimización bayesiana, con la función objetivo que queremos optimizar (en este caso, la variable val_loss)
tuner = keras_tuner.BayesianOptimization(
    build_model,
    objective = "val_loss",
    project_name='project_name2')

In [16]:
#entrenamos el modelo con el método search
tuner.search(X_train, X_train, epochs=50, validation_data=(X_test, X_test))

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82s/step - loss: 0.1029

Traceback (most recent call last):
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usua

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Usuario\anaconda3\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
             

In [25]:
#visualizamos los hiperparámetros que ha utilizado el modelo
tuner.search_space_summary()

Search space summary
Default search space size: 6
lr (Float)
{'default': 1e-05, 'conditions': [], 'min_value': 1e-05, 'max_value': 0.001, 'step': 10, 'sampling': 'log'}
batch_size (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 128, 'step': 2, 'sampling': 'log'}
act_hid (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'sigmoid', 'tanh'], 'ordered': False}
act_out (Choice)
{'default': 'linear', 'conditions': [], 'values': ['linear', 'sigmoid'], 'ordered': False}
loss_funct (Choice)
{'default': 'mse', 'conditions': [], 'values': ['mse', 'mae', 'mape'], 'ordered': False}
units (Int)
{'default': None, 'conditions': [], 'min_value': 30, 'max_value': 70, 'step': 5, 'sampling': 'linear'}


In [26]:
#visualizamos los mejores resultados
tuner.results_summary()

Results summary
Results in .\project_name
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 01 summary
Hyperparameters:
lr: 0.001
batch_size: 128
act_hid: relu
act_out: linear
loss_funct: mse
units: 45
Score: 0.04878459870815277

Trial 07 summary
Hyperparameters:
lr: 0.0001
batch_size: 128
act_hid: tanh
act_out: linear
loss_funct: mse
units: 65
Score: 0.0974559411406517

Trial 05 summary
Hyperparameters:
lr: 1e-05
batch_size: 32
act_hid: tanh
act_out: sigmoid
loss_funct: mse
units: 45
Score: 0.17710109055042267

Trial 06 summary
Hyperparameters:
lr: 0.0001
batch_size: 64
act_hid: tanh
act_out: linear
loss_funct: mse
units: 35
Score: 0.20602832734584808

Trial 00 summary
Hyperparameters:
lr: 0.001
batch_size: 32
act_hid: relu
act_out: sigmoid
loss_funct: mae
units: 30
Score: 0.3241247534751892

Trial 02 summary
Hyperparameters:
lr: 1e-05
batch_size: 128
act_hid: relu
act_out: linear
loss_funct: mae
units: 70
Score: 0.3588324189186096

Trial 03 summary
Hyperparame

In [27]:
#obtenemos el mejor modelo, y el listado de los hiperparámetros óptimos
best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hps.values

{'lr': 0.001,
 'batch_size': 128,
 'act_hid': 'relu',
 'act_out': 'linear',
 'loss_funct': 'mse',
 'units': 45}